In [1]:
import os
import pandas as pd
import ast
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, f1_score, roc_auc_score, recall_score

In [2]:
# Veri setini yükle
df = pd.read_csv(r"file_path")

# 'sequence' ve 'One_Hot_Encoded' kolonlarını çıkar
df = df.drop(["sequence", "numeric_sequence"], axis=1)

# AAC ve DPC sütunlarını işle
for col in ["AAC", "DPC"]:
    col_df = df[col].apply(lambda x: pd.Series(ast.literal_eval(x)))
    col_df = col_df.rename(columns=lambda x: f"{col}_{str(x)}")
    df = pd.concat([df.drop(col, axis=1), col_df], axis=1)

# Sütun isimleri string formatına dönüştürülür ve sayısal veri türüne çevirilir.
df.columns = [str(col) for col in df.columns]
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Eksik değerleri doldur (NaN değerler için)
df.fillna(0, inplace=True)

# Özellikleri ve hedef değişkeni ayır
X = df.drop("anticancer", axis=1)
y = df["anticancer"]

# Veri setini eğitim ve test setlerine ayır
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Özellik ölçeklendirme
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [3]:
# Hiperparametre optimizasyonundan elde edilen en iyi parametreler
best_params = {
    'bootstrap': True,
    'criterion': 'gini',
    'max_depth': 18,
    'max_features': 'sqrt',
    'min_samples_leaf': 2,
    'min_samples_split': 10,
    'n_estimators': 50,
    'oob_score': True,
    'warm_start': True
}

# En iyi parametrelerle modeli oluştur
optimized_rf_model = RandomForestClassifier(**best_params, random_state=42)

# Modeli eğit
optimized_rf_model.fit(X_train, y_train)

# Test verisi üzerinde tahmin yap
optimized_rf_predictions = optimized_rf_model.predict(X_test)

# Performans metriklerini hesapla ve yazdır
accuracy = accuracy_score(y_test, optimized_rf_predictions)
cm = confusion_matrix(y_test, optimized_rf_predictions)
sensitivity = recall_score(y_test, optimized_rf_predictions)
specificity = cm[0, 0] / (cm[0, 0] + cm[0, 1])
precision = precision_score(y_test, optimized_rf_predictions)
f1 = f1_score(y_test, optimized_rf_predictions)
roc_auc = roc_auc_score(y_test, optimized_rf_model.predict_proba(X_test)[:, 1])

print(f"Accuracy: {accuracy}")
print(f"Sensitivity: {sensitivity}")
print(f"Specificity: {specificity}")
print(f"Precision: {precision}")
print(f"F1 Score: {f1}")
print(f"ROC AUC Score: {roc_auc}")

Accuracy: 0.8936170212765957
Sensitivity: 0.8392857142857143
Specificity: 0.9294117647058824
Precision: 0.8867924528301887
F1 Score: 0.8623853211009174
ROC AUC Score: 0.928361344537815


In [4]:
# Modeli kaydet
joblib.dump(optimized_rf_model, 'RF_acc_dpc_bestModel.joblib')

['RF_acc_dpc_bestModel.joblib']

In [6]:
# 10 katlı Stratified Cross-Validation (Çapraz Doğrulama) başlatılır
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Metrikler ve tahminler için boş listeler başlatılır
all_scores = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': [], 'roc_auc': [], 'sensitivity': [], 'specificity': []}
all_predictions = []
all_true_labels = []

# warm_start False olarak ayarlanır, True için doğrulama metodu uyarı veriyor ve sonuçlar benzer çıkıyor
optimized_rf_model.set_params(warm_start=False)

# Çapraz doğrulama işlemi gerçekleştirilir
for train_index, test_index in cv.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Model eğitilir ve tahminler yapılır
    optimized_rf_model.fit(X_train, y_train)
    predictions = optimized_rf_model.predict(X_test)
    probabilities = optimized_rf_model.predict_proba(X_test)[:, 1]

    # Tahminler ve gerçek etiketler saklanır
    all_predictions.extend(predictions)
    all_true_labels.extend(y_test)

    # Performans metrikleri hesaplanır ve saklanır
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    roc_auc = roc_auc_score(y_test, probabilities)

    cm = confusion_matrix(y_test, predictions)
    sensitivity = recall
    specificity = cm[0, 0] / (cm[0, 0] + cm[0, 1]) if (cm[0, 0] + cm[0, 1]) != 0 else 0

    all_scores['accuracy'].append(accuracy)
    all_scores['precision'].append(precision)
    all_scores['recall'].append(recall)
    all_scores['f1_score'].append(f1)
    all_scores['roc_auc'].append(roc_auc)
    all_scores['sensitivity'].append(sensitivity)
    all_scores['specificity'].append(specificity)

# Skorlar ve tahminler DataFrame'lere dönüştürülür
scores_df = pd.DataFrame(all_scores)
predictions_df = pd.DataFrame({'True Labels': all_true_labels, 'Predictions': all_predictions})

# DataFrame'ler Excel dosyalarına kaydedilir
scores_df.to_excel("model_scores.xlsx", index=False)
predictions_df.to_excel("model_predictions.xlsx", index=False)